# KIS 동적 자산배분 자동화 - 단계별 테스트

모의투자 환경에서 전략을 테스트하고 자동화합니다.

In [3]:
# Cell [1]: 환경 설정 및 클라이언트 초기화
import sys
sys.path.insert(0, '/root')

from kis_trading_system import KISClient, Config, StrategyEngine, StateManager, TradingSystem
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

# 설정 확인
print("="*70)
print("🚀 KIS 자동화 거래 시스템")
print("="*70)
print(f"모드: {'🎭 모의투자' if Config.PAPER_TRADING else '💰 실거래'}")
print(f"App Key: {Config.KIS_APP_KEY[:20]}...")
print(f"사용 ETF: {len(Config.STRATEGY_ETFS)}개")
print("="*70)

# 클라이언트 초기화 (새 버전은 이렇게 간단!)
client = KISClient(paper_trading=Config.PAPER_TRADING)

print("\n✅ 클라이언트 초기화 완료")

2026-08-30 15:43:44 - KISTradingSystem - INFO - KIS 클라이언트 초기화 (모의투자: True)


🚀 KIS 자동화 거래 시스템
모드: 🎭 모의투자
App Key: PS4fOddELCfnyMkdLIaF...
사용 ETF: 7개

✅ 클라이언트 초기화 완료


In [5]:
# Cell [2]: kis_trading_system.py 파일 수정 (파라미터 자동 수정)
# 이 셀은 한 번만 실행하면 됩니다

file_path = "kis_trading_system.py"

with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

# 수정 1: FUND_STKQTY_DVSN → FUND_STTL_ICLD_YN
original_content = content
content = content.replace("'FUND_STKQTY_DVSN': '01'", "'FUND_STTL_ICLD_YN': 'N'")

# 수정 2: PRCS_DVSN '00' → '01'
content = content.replace("'PRCS_DVSN': '00'", "'PRCS_DVSN': '01'")

if content != original_content:
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print("✅ kis_trading_system.py 파라미터 수정 완료!")
    print("   - FUND_STKQTY_DVSN → FUND_STTL_ICLD_YN")
    print("   - PRCS_DVSN: '00' → '01'")
else:
    print("✓ 파일이 이미 최신 상태입니다")

✓ 파일이 이미 최신 상태입니다


In [6]:
# Cell [3]: 토큰 획득
print("🔐 KIS API 인증 중...")

if client.authenticate():
    print(f"✅ 인증 성공")
    print(f"   토큰: {client.token[:30]}...")
else:
    print("❌ 인증 실패")

2026-08-30 15:43:51 - KISTradingSystem - INFO - 🔐 KIS API 인증 중...
2026-08-30 15:43:51 - KISTradingSystem - INFO - ✅ 인증 성공 (토큰: eyJ0eXAiOi...)


🔐 KIS API 인증 중...
✅ 인증 성공
   토큰: eyJ0eXAiOiJKV1QiLCJhbGciOiJIUz...


In [7]:
# Cell [4]: 계좌 잔액 조회
print("💰 계좌 잔액 조회 중...")

balance = client.get_balance()

if balance:
    print(f"✅ 잔액 조회 성공")
    print(f"   보유 종목: {len(balance['holdings'])}개")
    if balance['holdings']:
        for code, qty in balance['holdings'].items():
            print(f"     {code}: {qty}주")
else:
    print("❌ 잔액 조회 실패")

2026-08-30 15:43:52 - KISTradingSystem - INFO - ✅ 잔액 조회: 보유 종목 1개


💰 계좌 잔액 조회 중...
✅ 잔액 조회 성공
   보유 종목: 1개
     005930: 1주


In [8]:
# Cell [5]: ETF 가격 조회
print("📊 ETF 가격 조회 중...")
print()

prices = {}
for etf_code, kis_code in Config.ETF_CODES.items():
    price = client.get_price(kis_code)
    if price:
        prices[etf_code] = price
        print(f"  {etf_code:6} ({kis_code}): ₩{price:,.0f}")

print(f"\n✅ {len(prices)}개 ETF 가격 조회 완료")

2026-08-30 15:43:54 - KISTradingSystem - ERROR - ❌ 가격 조회 실패 (305720): 500 Server Error: Internal Server Error for url: https://openapivts.koreainvestment.com:29443/uapi/domestic-stock/v1/quotations/inquire-price?fid_cond_mrkt_div_code=J&fid_input_iscd=305720


📊 ETF 가격 조회 중...

  ETF_02 (091160): ₩124,255
  ETF_07 (279530): ₩17,835


2026-08-30 15:43:55 - KISTradingSystem - ERROR - ❌ 가격 조회 실패 (143850): 500 Server Error: Internal Server Error for url: https://openapivts.koreainvestment.com:29443/uapi/domestic-stock/v1/quotations/inquire-price?fid_cond_mrkt_div_code=J&fid_input_iscd=143850
2026-08-30 15:43:55 - KISTradingSystem - ERROR - ❌ 가격 조회 실패 (132030): 500 Server Error: Internal Server Error for url: https://openapivts.koreainvestment.com:29443/uapi/domestic-stock/v1/quotations/inquire-price?fid_cond_mrkt_div_code=J&fid_input_iscd=132030


  ETF_09 (133690): ₩179,490


2026-08-30 15:43:55 - KISTradingSystem - ERROR - ❌ 가격 조회 실패 (114800): 500 Server Error: Internal Server Error for url: https://openapivts.koreainvestment.com:29443/uapi/domestic-stock/v1/quotations/inquire-price?fid_cond_mrkt_div_code=J&fid_input_iscd=114800



✅ 3개 ETF 가격 조회 완료


In [9]:
# Cell [6]: 전략 신호 계산
print("📈 전략 신호 계산 중...")
print()

# 임시 S&P500 신호 (실제로는 데이터에서 계산)
signal = {
    'regime': 'growth',
    'ratio': 1.01,
    'current': 5000,
    'ma60': 4950
}

strategy = StrategyEngine(Config.STRATEGY_ETFS)
print(f"현재 레짐: {signal['regime'].upper()}")
print(f"S&P500 비율: {signal['ratio']:.4f}")
print(f"현재가: {signal['current']}, 60일 MA: {signal['ma60']}")

2026-08-30 15:43:56 - KISTradingSystem - INFO - 전략 엔진 초기화


📈 전략 신호 계산 중...

현재 레짐: GROWTH
S&P500 비율: 1.0100
현재가: 5000, 60일 MA: 4950


In [10]:
# Cell [7]: 목표 자산배분 계산
print("🎯 목표 자산배분 계산 중...")
print()

target_allocation = strategy.get_target_allocation(signal)

if target_allocation:
    print(f"레짐 '{signal['regime']}' 목표 배분:")
    print()
    for etf_code, weight in sorted(target_allocation.items()):
        bar_length = int(weight * 40)
        bar = '█' * bar_length
        print(f"  {etf_code}: {bar} {weight*100:.1f}%")

2026-08-30 15:43:59 - KISTradingSystem - INFO - 타겟 배분 (growth): {'ETF_09': 0.2, 'ETF_10': 0.25, 'ETF_04': 0.15, 'ETF_02': 0.15, 'ETF_07': 0.15, 'ETF_22': 0.1, 'ETF_29': 0.0}


🎯 목표 자산배분 계산 중...

레짐 'growth' 목표 배분:

  ETF_02: ██████ 15.0%
  ETF_04: ██████ 15.0%
  ETF_07: ██████ 15.0%
  ETF_09: ████████ 20.0%
  ETF_10: ██████████ 25.0%
  ETF_22: ████ 10.0%
  ETF_29:  0.0%


In [12]:
# Cell [8]: 필요한 주문 계산
print("📋 필요한 주문 계산 중...")
print()

# 현재 포지션 (임시)
current_positions = {}
portfolio_value = balance['total_value']

orders = strategy.calculate_orders(
    current_positions=current_positions,
    portfolio_value=portfolio_value,
    prices=prices,
    target_allocation=target_allocation
)

if orders:
    print(f"계산된 주문: {len(orders)}개")
    print()
    for order in orders:
        side_emoji = "📈" if order['side'] == 'buy' else "📉"
        print(f"  {side_emoji} {order['etf_code']}: {order['quantity']}주 {order['side']} @ ₩{order['current_price']:,.0f}")
else:
    print("실행할 주문 없음")

2026-08-30 15:44:04 - KISTradingSystem - INFO - 계산된 주문: 0개


📋 필요한 주문 계산 중...

실행할 주문 없음


In [13]:
# Cell [9]: 시스템 전체 실행 (선택사항)
print("🚀 전체 시스템 실행")
print()

# 전체 자동화 시스템 사용
system = TradingSystem()
success = system.run()

if success:
    print("\n✅ 거래 완료")
else:
    print("\n❌ 거래 실패")

2026-08-30 15:44:06 - KISTradingSystem - INFO - KIS 클라이언트 초기화 (모의투자: True)
2026-08-30 15:44:06 - KISTradingSystem - INFO - 전략 엔진 초기화
2026-08-30 15:44:06 - KISTradingSystem - INFO - ======================================================================
2026-08-30 15:44:06 - KISTradingSystem - INFO - 🚀 KIS 자동화 거래 시스템 시작
2026-08-30 15:44:06 - KISTradingSystem - INFO - 📱 모드: 🎭 모의투자
2026-08-30 15:44:06 - KISTradingSystem - INFO - 🕐 실행시간: 15:45 (한국시간)
2026-08-30 15:44:06 - KISTradingSystem - INFO - 📊 사용 ETF: 7개
2026-08-30 15:44:06 - KISTradingSystem - INFO - ======================================================================
2026-08-30 15:44:06 - KISTradingSystem - INFO - 📍 거래 주기 시작
2026-08-30 15:44:06 - KISTradingSystem - INFO - 🔐 KIS API 인증 중...
2026-08-30 15:44:06 - KISTradingSystem - INFO - ✅ 인증 성공 (토큰: eyJ0eXAiOi...)
2026-08-30 15:44:06 - KISTradingSystem - INFO - ✅ 잔액 조회: 보유 종목 1개
2026-08-30 15:44:06 - KISTradingSystem - INFO - 현재 포지션: {}


🚀 전체 시스템 실행



2026-08-30 15:44:07 - KISTradingSystem - ERROR - ❌ 가격 조회 실패 (305720): 500 Server Error: Internal Server Error for url: https://openapivts.koreainvestment.com:29443/uapi/domestic-stock/v1/quotations/inquire-price?fid_cond_mrkt_div_code=J&fid_input_iscd=305720
2026-08-30 15:44:07 - KISTradingSystem - ERROR - ❌ 가격 조회 실패 (133690): 500 Server Error: Internal Server Error for url: https://openapivts.koreainvestment.com:29443/uapi/domestic-stock/v1/quotations/inquire-price?fid_cond_mrkt_div_code=J&fid_input_iscd=133690
2026-08-30 15:44:07 - KISTradingSystem - ERROR - ❌ 가격 조회 실패 (143850): 500 Server Error: Internal Server Error for url: https://openapivts.koreainvestment.com:29443/uapi/domestic-stock/v1/quotations/inquire-price?fid_cond_mrkt_div_code=J&fid_input_iscd=143850
2026-08-30 15:44:07 - KISTradingSystem - ERROR - ❌ 가격 조회 실패 (132030): 500 Server Error: Internal Server Error for url: https://openapivts.koreainvestment.com:29443/uapi/domestic-stock/v1/quotations/inquire-price?fid_cond_mr


✅ 거래 완료


In [16]:
# APScheduler 설치
!pip install apscheduler --break-system-packages
print("✅ APScheduler 설치 완료")

zsh:1: command not found: pip
✅ APScheduler 설치 완료


In [17]:
# Cell [10]: 자동화 스케줄러 설정 및 시작
print("🕐 자동화 스케줄러 설정 중...")
print()

# kis_trading_system.py는 이미 현재 폴더에 있으므로 바로 import
from kis_trading_system import setup_scheduler

# 스케줄러 시작
scheduler = setup_scheduler(system)

if scheduler:
    print("✅ 스케줄러 시작 완료!")
    print(f"📅 매일 15:45 (한국시간)에 자동 실행됩니다")
    print()
    print("🔔 현재 상태:")
    print(f"   - 모드: {'🎭 모의투자' if Config.PAPER_TRADING else '💰 실거래'}")
    print(f"   - 실행 시각: {Config.TRADING_TIME} KST")
    print(f"   - 상태: 🟢 활성화")
else:
    print("❌ 스케줄러 설정 실패")

2026-08-30 15:46:40 - KISTradingSystem - WARNING - APScheduler 미설치 - 스케줄링 미활성화
2026-08-30 15:46:40 - KISTradingSystem - WARNING - 설치: pip install apscheduler


🕐 자동화 스케줄러 설정 중...

❌ 스케줄러 설정 실패


In [19]:
# Cell [11]: 스케줄러 모니터링 & 거래 이력 확인
print("📊 스케줄러 모니터링")
print()

# 스케줄러 상태 확인
if scheduler and scheduler.running:
    print("✅ 스케줄러 상태: 🟢 활성화 중")
    print()
    print("📋 스케줄된 작업:")
    for job in scheduler.get_jobs():
        print(f"   - {job.name}")
        print(f"     ID: {job.id}")
        print(f"     다음 실행: {job.next_run_time}")
    print()
else:
    print("⚠️ 스케줄러가 비활성화 상태입니다")

print("\n📁 거래 이력 확인:")
# 저장된 거래 기록 확인
import json
from pathlib import Path

history_file = Path(Config.HISTORY_FILE)
if history_file.exists():
    with open(history_file, 'r', encoding='utf-8') as f:
        history = json.load(f)
    
    if history:
        print(f"총 거래: {len(history)}건")
        print("\n최근 거래 (최대 3건):")
        for trade in history[-3:]:
            print(f"   - {trade['etf_code']}: {trade['quantity']}주 {trade['side'].upper()} @ ₩{trade['price']:,.0f}")
    else:
        print("아직 거래 기록이 없습니다")
else:
    print("거래 이력 파일이 없습니다")

📊 스케줄러 모니터링

⚠️ 스케줄러가 비활성화 상태입니다

📁 거래 이력 확인:
거래 이력 파일이 없습니다
